# 🔄 Day 04a: Page Replacement — FIFO, LRU, Optimal & Belady's Anomaly

---

## 🎯 What You'll Master Today
- FIFO, LRU, and Optimal page replacement algorithms
- Clock (Second Chance) algorithm
- Belady's anomaly — more frames ≠ fewer faults?
- Algorithm comparison with the same reference string

---

```
╔══════════════════════════════════════════════════════════════╗
║           PAGE REPLACEMENT ALGORITHM OVERVIEW                ║
╠════════════════╦═════════════════════════════════════════════╣
║  FIFO          ║  Evict oldest page. Simple but not smart   ║
║  LRU           ║  Evict least recently used. Good but costly║
║  Optimal (OPT) ║  Evict page used farthest in future. Ideal║
║  Clock         ║  FIFO + reference bit = cheap LRU approx   ║
╠════════════════╩═════════════════════════════════════════════╣
║  OPT = theoretical best (can't implement — needs future)    ║
║  LRU = practical best (expensive to implement exactly)      ║
║  Clock = what most real OSes actually use                    ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. FIFO Page Replacement
# ============================================

from collections import deque

def fifo_replacement(reference_string: list[int], num_frames: int) -> int:
    frames = deque(maxlen=num_frames)
    page_set = set()
    faults = 0
    
    print(f"  FIFO ({num_frames} frames)")
    print(f"  {'Ref':<5} {'Frames':<25} {'Status'}")
    print("  " + "─" * 45)
    
    for page in reference_string:
        if page not in page_set:
            faults += 1
            if len(frames) == num_frames:
                evicted = frames.popleft()
                page_set.remove(evicted)
            frames.append(page)
            page_set.add(page)
            status = "FAULT 📥"
        else:
            status = "HIT   ✅"
        
        print(f"  {page:<5} {str(list(frames)):<25} {status}")
    
    print(f"\n  Total Page Faults: {faults}")
    return faults


ref = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]
fifo_replacement(ref, 3)

In [ ]:
# ============================================
# 2. LRU Page Replacement
# ============================================

from collections import OrderedDict

def lru_replacement(reference_string: list[int], num_frames: int) -> int:
    frames = OrderedDict()  # order = access recency
    faults = 0
    
    print(f"  LRU ({num_frames} frames)")
    print(f"  {'Ref':<5} {'Frames':<25} {'Status'}")
    print("  " + "─" * 45)
    
    for page in reference_string:
        if page in frames:
            frames.move_to_end(page)  # update recency
            status = "HIT   ✅"
        else:
            faults += 1
            if len(frames) >= num_frames:
                frames.popitem(last=False)  # evict LRU (oldest)
            frames[page] = True
            status = "FAULT 📥"
        
        print(f"  {page:<5} {str(list(frames.keys())):<25} {status}")
    
    print(f"\n  Total Page Faults: {faults}")
    return faults


ref = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]
lru_replacement(ref, 3)

In [ ]:
# ============================================
# 3. Optimal (OPT) Page Replacement
# ============================================

def optimal_replacement(reference_string: list[int], num_frames: int) -> int:
    frames = []
    faults = 0
    
    print(f"  Optimal ({num_frames} frames)")
    print(f"  {'Ref':<5} {'Frames':<25} {'Status'}")
    print("  " + "─" * 45)
    
    for i, page in enumerate(reference_string):
        if page in frames:
            status = "HIT   ✅"
        else:
            faults += 1
            if len(frames) < num_frames:
                frames.append(page)
            else:
                # Evict page used farthest in future
                farthest = -1
                evict_idx = 0
                for j, f in enumerate(frames):
                    try:
                        next_use = reference_string[i+1:].index(f)
                    except ValueError:
                        next_use = float('inf')  # never used again
                    if next_use > farthest:
                        farthest = next_use
                        evict_idx = j
                frames[evict_idx] = page
            status = "FAULT 📥"
        
        print(f"  {page:<5} {str(frames[:]):<25} {status}")
    
    print(f"\n  Total Page Faults: {faults}")
    return faults


ref = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]
optimal_replacement(ref, 3)

In [ ]:
# ============================================
# 4. Comparison + Belady's Anomaly
# ============================================

print("  ═" * 25)
print("  ALGORITHM COMPARISON (same reference string, 3 frames)")
print("  ═" * 25)

ref = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2, 1, 2, 0, 1, 7, 0, 1]

# Suppress individual prints for summary
import io, sys

results = {}
for name, func in [("FIFO", fifo_replacement), ("LRU", lru_replacement), ("OPT", optimal_replacement)]:
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    results[name] = func(ref, 3)
    sys.stdout = old_stdout

print("\n  Results:")
for name, faults in results.items():
    bar = '█' * faults + '░' * (20 - faults)
    print(f"    {name:<8} {bar} {faults} faults")

print("\n" + "  ═" * 25)
print("  BELADY'S ANOMALY — FIFO with more frames = MORE faults!")
print("  ═" * 25)

# Classic Belady's example
belady_ref = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]
print(f"  Reference: {belady_ref}\n")

for frames in [3, 4]:
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    faults = fifo_replacement(belady_ref, frames)
    sys.stdout = old_stdout
    print(f"    FIFO with {frames} frames: {faults} faults")

print("\n  😱 More frames (4) → MORE faults than 3 frames!")
print("  💡 Belady's anomaly only affects FIFO, never LRU or OPT (stack algorithms).")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Why can't OPT be implemented? | Needs future knowledge. Used as benchmark to compare other algorithms |
| 2 | LRU implementation options? | Counter-based (timestamp), stack-based, or approximate with reference bits |
| 3 | What is Belady's anomaly? | FIFO: more frames can cause MORE page faults. Doesn't affect stack algorithms |
| 4 | What is the Clock algorithm? | FIFO + reference bit. Give second chance if bit=1 (reset to 0), evict if bit=0 |
| 5 | Which does Linux use? | LRU approximation (Clock variant) with active/inactive page lists |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • OPT < LRU < FIFO in fault count (usually)            │
## │  • FIFO: simple but suffers Belady's anomaly             │
## │  • LRU: good but expensive (needs tracking per access)   │
## │  • Clock: practical LRU approximation used in real OSes  │
## │  • Stack algorithms (LRU, OPT) immune to Belady's       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **File Organization** — file allocation methods and inodes